# **TP2 - Bivariate Analysis**

<b>Exploratory Data Analysis & Unsupervised Learning </b><br>
**Course: Dr. HAS Sothea** <br>
**TP: Ms. UANN Sreyvi** <br>
**Student: NOB Sreynich (ID: e20220741)**

-------


**Objective**: To equip you all with the skills to analyze and interpret the relationship between two variables. You will explore these relationships as a function of time using the [`Gapminder`](https://pypi.org/project/gapminder/) dataset.

---------

> **The `Jupyter Notebook` for this TP can be downloaded here: [TP2-Bivariate-Gapminder](https://hassothea.github.io/I5_EDA_ITC/TPs/TP2_Bivariate_Analysis.ipynb)**.

-------


In [ ]:
# Useful packages for Bivariate Analysis
# %pip install gapminder pingouin plotly seaborn matplotlib scipy
from gapminder import gapminder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pingouin as pg
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

print(f"* Gapminder dataset loaded: {gapminder.shape[0]} rows, {gapminder.shape[1]} columns")
print(f"* Years covered: {sorted(gapminder['year'].unique())}")
gapminder.head()


# **1. Pearson and Spearman's correlations**

**a.** Compute Pearson correlation matrix of the three quantitative variables on year $1952$, $1987$ and then $2007$ using `pd.corr()`. Give a brief intuition of the relationship between these variables.


In [ ]:
data1952 = gapminder.query("year == 1952") # Select rows such that "year == 1952"
data1987 = gapminder.query("year == 1987")
data2007 = gapminder.query("year == 2007")


In [ ]:
quan_vars = ['lifeExp', 'gdpPercap', 'pop']
data1952[quan_vars].corr().style.background_gradient(cmap='hot')


In [ ]:
data1987[quan_vars].corr().style.background_gradient(cmap='hot')


In [ ]:
data2007[quan_vars].corr().style.background_gradient(cmap='hot')


### **Description & Intuition (Pearson Correlation):**

**Pearson's Correlation Coefficient ($r \in [-1, 1]$)** measures the strength and direction of a **linear relationship** between two quantitative variables:
$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

Analyzing the Pearson correlation matrices for $1952$, $1987$, and $2007$:

1. **`lifeExp` vs. `gdpPercap` (Health & Wealth)**:
   - **1952**: $r = 0.2780$ (Weak-to-moderate positive linear correlation).
   - **1987**: $r = 0.7499$ (Strong positive linear correlation).
   - **2007**: $r = 0.6787$ (Strong positive linear correlation).
   - *Intuition*: Higher GDP per capita empowers governments and individuals to invest in modern healthcare facilities, clean water, adequate nutrition, vaccines, and disease control, leading to longer life expectancy. In 1952, the linear correlation was surprisingly low ($r = 0.278$) due to extreme outliers (e.g., Kuwait), but strengthened dramatically by 1987 and 2007 as healthcare and economic gains became more globally widespread.

2. **`pop` vs. `lifeExp` & `pop` vs. `gdpPercap` (Demographics vs. Health/Wealth)**:
   - In all three years, the correlation between population size and life expectancy is practically zero ($r = -0.0027$ in 1952, $0.0331$ in 1987, $0.0476$ in 2007).
   - Similarly, the correlation between population and GDP per capita is negligible ($r = -0.0253$ in 1952, $-0.0517$ in 1987, $-0.0557$ in 2007).
   - *Intuition*: A country's total population has **no linear relationship** with its health standards or per capita income. The most populous nations include both high-income countries (USA) and developing nations (India, China), while small-population nations range from extremely rich (Norway, Switzerland, Kuwait) to very poor.

-----


**b.** Compute Spearman's Rank Correlation of the previous columns in 1952, 1987 and 2007. What do you observe?


In [ ]:
data1952[quan_vars].corr(method='spearman').style.background_gradient(cmap='hot')


In [ ]:
data1987[quan_vars].corr(method='spearman').style.background_gradient(cmap='hot')


In [ ]:
data2007[quan_vars].corr(method='spearman').style.background_gradient(cmap='hot')


> ### **What Do You Observe? (Pearson vs. Spearman)**
>
> Comparing `lifeExp` vs. `gdpPercap` across the two correlation methods:
>
> | Metric | Year 1952 | Year 1987 | Year 2007 |
> | :--- | :---: | :---: | :---: |
> | **Pearson ($r$)** | **0.2780** | **0.7499** | **0.6787** |
> | **Spearman ($\rho$)** | **0.7687** | **0.8925** | **0.8566** |
> | **Difference ($\rho - r$)** | **+0.4907** | **+0.1426** | **+0.1779** |
>
> **Key Observations:**
> 1. **Spearman's rank correlation is higher than Pearson's correlation across all years**, especially in 1952 where Spearman ($\rho = 0.769$) is nearly triple Pearson ($r = 0.278$)!
> 2. **Linearity vs. Monotonicity**: Pearson strictly evaluates whether the data falls along a straight line ($y = ax + b$). Spearman evaluates **monotonicity** (whether $y$ increases as $x$ increases, regardless of curvature) by ranking the values.
> 3. **Non-linear Relationship (The Preston Curve)**: The relationship between GDP per capita and life expectancy is **concave / logarithmic**—initial economic gains yield huge increases in life expectancy, but returns diminish at higher income levels. Spearman captures this monotonic relationship much more faithfully than Pearson.
> 4. **Robustness to Outliers**: Spearman is resistant to extreme outliers because values are converted to ranks ($1, 2, \dots, n$). In contrast, Pearson is heavily distorted by extreme values like Kuwait in 1952.
>
> -----


**c.** From the previous result, pick the most interesting pair of variables and plot a graph illustrating their relationship on each year, using proper axis scaling and title.


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'GDP (log) vs Life Expectancy ({yr})' for yr in [1952, 1987, 2007]]
)

# 1952 trace
fig.add_trace(
    go.Scatter(
        x=data1952['gdpPercap'],
        y=data1952['lifeExp'],
        mode='markers',
        name='Year 1952',
        text=data1952['country'],
        marker=dict(size=7, color='#1f77b4', opacity=0.75)
    ),
    row=1, col=1
)

# 1987 trace
fig.add_trace(
    go.Scatter(
        x=data1987['gdpPercap'],
        y=data1987['lifeExp'],
        mode='markers',
        name='Year 1987',
        text=data1987['country'],
        marker=dict(size=7, color='#2ca02c', opacity=0.75)
    ),
    row=1, col=2
)

# 2007 trace
fig.add_trace(
    go.Scatter(
        x=data2007['gdpPercap'],
        y=data2007['lifeExp'],
        mode='markers',
        name='Year 2007',
        text=data2007['country'],
        marker=dict(size=7, color='#9467bd', opacity=0.75)
    ),
    row=1, col=3
)

fig.update_layout(
    width=1100,
    height=420,
    title='GDP per capita (Log Scale) vs Life Expectancy (1952, 1987, 2007)',
    showlegend=True
)

fig.update_xaxes(type='log', col=1, row=1, title='GDP per capita ($ - log)')
fig.update_xaxes(type='log', col=2, row=1, title='GDP per capita ($ - log)')
fig.update_xaxes(type='log', col=3, row=1, title='GDP per capita ($ - log)')
fig.update_yaxes(col=1, row=1, title='Life Expectancy (years)')
fig.update_yaxes(col=2, row=1, title='Life Expectancy (years)')
fig.update_yaxes(col=3, row=1, title='Life Expectancy (years)')

fig.show()


**d.** Revisit your intuition of the correlation matrix in year 1952 from question **(a)**, can you see why we observed such a (poor) correlation in 1952?!

----


In [ ]:
# Investigating the 1952 Anomaly: The Kuwait Outlier
df1952 = gapminder.query("year == 1952").copy()
print("Top 5 Richest Countries in 1952 by GDP per capita:")
display(df1952.sort_values(by='gdpPercap', ascending=False)[['country', 'continent', 'gdpPercap', 'lifeExp', 'pop']].head(5))

# Pearson r with Kuwait vs. without Kuwait vs. log-transformed
r_with = df1952['lifeExp'].corr(df1952['gdpPercap'])
df1952_no_kuwait = df1952[df1952['country'] != 'Kuwait']
r_without = df1952_no_kuwait['lifeExp'].corr(df1952_no_kuwait['gdpPercap'])
r_log = df1952['lifeExp'].corr(np.log(df1952['gdpPercap']))

print(f"\n1952 Pearson r WITH Kuwait:       {r_with:.4f}")
print(f"1952 Pearson r WITHOUT Kuwait:    {r_without:.4f}  (nearly triples!)")
print(f"1952 Pearson r on log(gdpPercap): {r_log:.4f}")


> **Remark:** Pearson correlation matrix can summarize linear relationship between pairs of quantitative variables but it might be inacurate and influenced by 
> 
> - outliers, 
> - non-linearity, 
> - small sample size, 
> - confounding (causal) variables...


### **Conclusion:**

The poor Pearson correlation in 1952 ($r = 0.2780$) is primarily caused by an **extreme high-leverage outlier: Kuwait**.

1. **The Kuwait Outlier**: In 1952, Kuwait had an astronomical GDP per capita of **\$108,382** (due to immense oil discoveries combined with a tiny population of ~160,000 citizens). In comparison, Switzerland was the second richest country at only **\$14,734**. It mean that when  Kuwait had over **7 times the wealth** of the next richest nation!
2. **Modest Longevity**: Despite its massive wealth, Kuwait's life expectancy in 1952 was only **55.57 years** (modern sanitary and medical infrastructure had not yet fully developed).
3. **High Leverage Effect**: In a linear regression ($r$), this single data point sits at the extreme right ($x = 108,382, y = 55.6$) and acts as an enormous lever, pulling the regression line down and severely flattening its slope.
4. **Impact of Outlier Removal & Log Transformation**:
   - Removing Kuwait alone causes the 1952 Pearson correlation to leap from **$0.2780$ to $0.7466$** (almost tripling!).
   - Applying a logarithmic transformation $\log(\text{gdpPercap})$ raises the correlation to **$0.7478$**.
   - Spearman's rank correlation ($\rho = 0.7687$) was unaffected because Kuwait was simply assigned rank 142.


----


# **2. Visualization with more information**

`Scatterplot` is the primary graphic type for visualizing the relationship between two quantitative variables. Additionally, you can include other factors of interest using color, shape, size, facets, etc., depending on the type of those variables. 

------------

The following questions apply to the years 1952, 1987, and 2007:

**a.** Do you think health conditions and economies differ across continents? Compute some indicators to illustrate this.


In [ ]:
# Ensure data variables are defined
from gapminder import gapminder
if 'data1952' not in locals():
    data1952 = gapminder.query("year == 1952")
if 'data1987' not in locals():
    data1987 = gapminder.query("year == 1987")
if 'data2007' not in locals():
    data2007 = gapminder.query("year == 2007")

# Indicator: eta-squared coefficients in 1952
import pingouin as pg
eta2_life_52 = pg.anova(
    data1952, 
    between='continent',
    dv='lifeExp'
)
print("eta^2 for continent vs lifeExp in 1952:", eta2_life_52['np2'].values[0])
eta2_life_52


In [ ]:
eta2_gdp_52 = pg.anova(
    data1952, 
    between='continent',
    dv='gdpPercap'
)
print("eta^2 for continent vs gdpPercap in 1952:", eta2_gdp_52['np2'].values[0])
eta2_gdp_52


In [ ]:
# Descriptive indicators across continents for 1952, 1987, and 2007
indicator_list = []
for yr, df_yr in [(1952, data1952), (1987, data1987), (2007, data2007)]:
    summary = df_yr.groupby('continent').agg(
        count=('country', 'count'),
        lifeExp_mean=('lifeExp', 'mean'),
        lifeExp_median=('lifeExp', 'median'),
        lifeExp_std=('lifeExp', 'std'),
        gdp_median=('gdpPercap', 'median'),
        gdp_mean=('gdpPercap', 'mean')
    ).round(2)
    summary['year'] = yr
    indicator_list.append(summary.reset_index())

all_indicators = pd.concat(indicator_list, ignore_index=True)
print("=== Mean Life Expectancy Across Continents ===")
display(all_indicators.pivot(index='continent', columns='year', values='lifeExp_mean'))

print("\n=== Median GDP per Capita (USD) Across Continents ===")
display(all_indicators.pivot(index='continent', columns='year', values='gdp_median'))

# Boxplots for Year 2007
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=data2007, x='continent', y='lifeExp', palette='Set2', ax=axes[0])
axes[0].set_title('Life Expectancy by Continent (2007)', fontweight='bold')
axes[0].set_ylabel('Life Expectancy (years)')

sns.boxplot(data=data2007, x='continent', y='gdpPercap', palette='Set2', ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('GDP per Capita by Continent (2007, Log Scale)', fontweight='bold')
axes[1].set_ylabel('GDP per Capita ($ - log)')

plt.tight_layout()
plt.show()


> ### **Description of Continental Disparities:**
>
> **Yes, health conditions and economies differ substantially across continents throughout 1952, 1987, and 2007.**
>
> 1. **Health Disparities (`lifeExp`)**:
>    - **Oceania & Europe**: Consistently boast the highest life expectancies globally (Europe mean: $64.4$ yrs in 1952 $\rightarrow$ $77.6$ yrs in 2007; Oceania: $69.3$ yrs in 1952 $\rightarrow$ $80.7$ yrs in 2007) with tight interquartile spreads, indicative of widespread access to universal healthcare and quality living conditions.
>    - **Africa**: Lags severely behind all other regions (mean: $39.1$ yrs in 1952 $\rightarrow$ $54.8$ yrs in 2007; median in 2007 is only $52.9$ yrs), with high internal dispersion due to regional armed conflicts and public health crises (e.g., malaria and HIV/AIDS epidemic).
>    - **Asia & Americas**: Show the fastest gains over the 55-year period (Asia surged from $46.3$ yrs in 1952 to $70.7$ yrs in 2007, adding more than 24 years of average life).
>
> 2. **Economic Disparities (`gdpPercap`)**:
>    - In 2007, median GDP per capita in Europe and Oceania exceeded **\$25,000**, compared to only **\$1,452** in Africa and **\$4,471** in Asia.
>    - The boxplots illustrate that the distributions of GDP per capita for Africa and Europe do not overlap at all, demonstrating persistent global structural economic divisions.
>
> -----


**c.** We will try to confirm this using Analysis of Variance (ANOVA). 

In 2007,

- Perform one way ANOVA on `lifeExp` and `continent` (you can use `f_oneway` from `scipy.stats` module).


In [ ]:
# One-way ANOVA: lifeExp across continents in 2007
groups_life_07 = [group['lifeExp'].values for _, group in data2007.groupby('continent')]
f_stat_life, p_val_life = stats.f_oneway(*groups_life_07)

print("=== One-Way ANOVA: `lifeExp` Across Continents (2007) ===")
print(f"F-statistic: {f_stat_life:.4f}")
print(f"p-value:     {p_val_life:.4e}")
if p_val_life < 0.05:
    print("Conclusion: Reject H0! There are statistically significant differences in mean life expectancy across continents.")


- Do the same with `gdpPercap` and `continent`.


In [ ]:
# One-way ANOVA: gdpPercap across continents in 2007
groups_gdp_07 = [group['gdpPercap'].values for _, group in data2007.groupby('continent')]
f_stat_gdp, p_val_gdp = stats.f_oneway(*groups_gdp_07)

print("=== One-Way ANOVA: `gdpPercap` Across Continents (2007) ===")
print(f"F-statistic: {f_stat_gdp:.4f}")
print(f"p-value:     {p_val_gdp:.4e}")
if p_val_gdp < 0.05:
    print("Conclusion: Reject H0! There are statistically significant differences in mean GDP per capita across continents.")

# Testing ANOVA Assumptions in 2007:
print("\n--- 1. Homogeneity of Variances (Levene's Test) ---")
lev_life = stats.levene(*groups_life_07)
lev_gdp = stats.levene(*groups_gdp_07)
print(f"lifeExp Levene test:   W = {lev_life.statistic:.4f}, p = {lev_life.pvalue:.4e} (Equal variance REJECTED)")
print(f"gdpPercap Levene test: W = {lev_gdp.statistic:.4f}, p = {lev_gdp.pvalue:.4e} (Equal variance REJECTED)")

print("\n--- 2. Normality of Residuals (Shapiro-Wilk Test per Continent) ---")
for cont, group in data2007.groupby('continent'):
    if len(group) >= 3:
        sh_life = stats.shapiro(group['lifeExp'])
        sh_gdp = stats.shapiro(group['gdpPercap'])
        print(f"{cont:10s} (n={len(group):2d}) | lifeExp p={sh_life.pvalue:.4f} | gdpPercap p={sh_gdp.pvalue:.4e}")


- Are these results reliable? Why?
>
> **No, the classical One-Way ANOVA results are NOT strictly reliable.**
>
> **Why?** One-Way ANOVA relies on three mandatory statistical assumptions:
> 1. **Homogeneity of Variances (Homoscedasticity)**: All continental groups must share approximately equal variance ($\sigma_1^2 = \dots = \sigma_k^2$).
>    - *Levene's test* yields $p = 4.94 \times 10^{-5}$ for `lifeExp` and $p = 4.31 \times 10^{-5}$ for `gdpPercap` ($p < 0.05$). The null hypothesis of equal variances is **firmly rejected**.
> 2. **Normality of Within-Group Distributions**: Data within each group should be normally distributed.
>    - *Shapiro-Wilk tests* show significant departures from normality ($p < 0.05$) for multiple continents, especially for `gdpPercap` in Africa ($p = 7.42 \times 10^{-9}$) and Asia, where distributions are heavily right-skewed.
> 3. **Group Size Balance**: Group sample sizes are heavily unbalanced (Africa: 52, Americas: 25, Asia: 33, Europe: 30, Oceania: 2). When group variances and sample sizes are both unequal, the classical $F$-statistic is prone to distorted Type I error rates.

- Propose ideas that might solve this problem or an alternative method.



In [ ]:
# Implementing the Proposed Solutions:
# 1. Kruskal-Wallis H-test (Non-parametric)
kw_life = stats.kruskal(*groups_life_07)
kw_gdp = stats.kruskal(*groups_gdp_07)
print("=== Solution 1: Kruskal-Wallis H-Test ===")
print(f"lifeExp:   H = {kw_life.statistic:.4f}, p = {kw_life.pvalue:.4e}")
print(f"gdpPercap: H = {kw_gdp.statistic:.4f}, p = {kw_gdp.pvalue:.4e}")

# 2. ANOVA on log-transformed gdpPercap
groups_log_gdp = [np.log10(g['gdpPercap'].values) for _, g in data2007.groupby('continent')]
f_log, p_log = stats.f_oneway(*groups_log_gdp)
print(f"\n=== Solution 2: ANOVA on log10(gdpPercap) ===")
print(f"F-statistic: {f_log:.4f}, p = {p_log:.4e}")

# 3. Welch's ANOVA
print("\n=== Solution 3: Welch's ANOVA (Robust to Heteroscedasticity) ===")
welch_life = pg.welch_anova(data=data2007, dv='lifeExp', between='continent')
welch_gdp = pg.welch_anova(data=data2007, dv='gdpPercap', between='continent')
display(welch_life)
display(welch_gdp)


**d.** We have previously described the relationship between population and life expectancy or GDP per capita. Now, illustrate the effect of population defining **size** of points according to variable `pop`.


In [ ]:
# 2.d Bubble chart: GDP per capita vs. Life Expectancy with bubble size proportional to population
fig = px.scatter(
    data2007,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    color='continent',
    hover_name='country',
    log_x=True,
    size_max=60,
    title='Gapminder 2007: Health vs. Wealth (Bubble Size = Population)',
    labels={'gdpPercap': 'GDP per Capita (USD, Log Scale)', 'lifeExp': 'Life Expectancy (years)', 'pop': 'Population'},
    template='plotly_white',
    width=950,
    height=550
)
fig.show()


**e.** Now, directly visualize the relation between `pop` and `lifeExp`, then the relation between `pop` and `gdpPercap`. Explain the resulting graphs.


In [ ]:
# 2.e Direct visualization: pop vs. lifeExp and pop vs. gdpPercap in 2007
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# pop vs lifeExp
sns.scatterplot(
    data=data2007,
    x='pop',
    y='lifeExp',
    hue='continent',
    alpha=0.8,
    s=70,
    ax=axes[0]
)
axes[0].set_xscale('log')
axes[0].set_title('Population vs. Life Expectancy (2007)', fontweight='bold')
axes[0].set_xlabel('Population (Log Scale)')
axes[0].set_ylabel('Life Expectancy (years)')
axes[0].grid(True, which="both", ls="--", alpha=0.4)

# pop vs gdpPercap
sns.scatterplot(
    data=data2007,
    x='pop',
    y='gdpPercap',
    hue='continent',
    alpha=0.8,
    s=70,
    ax=axes[1]
)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title('Population vs. GDP per Capita (2007, Log-Log Scale)', fontweight='bold')
axes[1].set_xlabel('Population (Log Scale)')
axes[1].set_ylabel('GDP per Capita (USD, Log Scale)')
axes[1].grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
plt.show()


> ### **Explanation of Graphs (Population vs. Health & Wealth):**
>
> 1. **Flat Horizontal Dispersion**: Both scatter plots show a **horizontal cloud of points with virtually no slope or linear trend**.
> 2. **Independence from Population Size**:
>    - Small nations (under 10 million people) span the entire range of health (from under 40 years to over 82 years) and wealth (from \$300 to over \$45,000).
>    - Megapopulated countries (China and India, with over 1 billion people) sit near the global intermediate levels of life expectancy (~65–73 years) and GDP per capita (\$2,500–\$5,000).
> 3. **Conclusion**:
>    - **Population size does not determine a nation's standard of living or life expectancy.** A country's health and wealth depend on institutional quality, healthcare systems, peace, governance, education, and capital investment is not demographic size.
>
> -----


# **3. Quantitative vs Qualitative**


**a.** We have seen how life expectancy and economy vary across continents in 2007 ([Slide 25 of the course](https://hassothea.github.io/EDA_ITC/Slides/Bivariate_Analysis.html#/quan.-vs-qual.-3)). Compute $\eta$-squared correlation between `continent` and `lifeExp`, then `continent` with `gdpPercap` in 1952, 1987 and 2007. 

- Do you find the results reasonable?


In [ ]:
# 3.a Compute Eta-squared (correlation ratio) for continent vs lifeExp and gdpPercap in 1952, 1987, and 2007
def compute_eta_squared(df, categorical_col, numerical_col):
    """
    Computes Correlation Ratio (Eta-squared):
    eta^2 = SS_between / SS_total
    """
    grand_mean = df[numerical_col].mean()
    ss_total = ((df[numerical_col] - grand_mean) ** 2).sum()
    group_stats = df.groupby(categorical_col)[numerical_col].agg(['count', 'mean'])
    ss_between = (group_stats['count'] * (group_stats['mean'] - grand_mean) ** 2).sum()
    return ss_between / ss_total

eta_records = []
for yr in [1952, 1987, 2007]:
    sub = gapminder.query(f"year == {yr}")
    eta_life = compute_eta_squared(sub, 'continent', 'lifeExp')
    eta_gdp = compute_eta_squared(sub, 'continent', 'gdpPercap')
    
    eta_records.append({
        'Year': yr,
        'eta^2 (continent vs. lifeExp)': round(eta_life, 4),
        'eta^2 (continent vs. gdpPercap)': round(eta_gdp, 4)
    })

eta_df = pd.DataFrame(eta_records)
print("=== Eta-squared (Correlation Ratio) Summary ===")
display(eta_df)


> ### **Do you find the results reasonable?**
>
> **Yes, the results are completely reasonable and statistically coherent:**
>
> 1. **Definition of $\eta^2$ (Correlation Ratio)**:
>    $$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}} = \frac{\sum_{k} n_k (\bar{y}_k - \bar{y})^2}{\sum_{i} (y_i - \bar{y})^2}$$
>    $\eta^2$ measures the proportion of total variance in the quantitative variable explained by continental group membership ($0 \le \eta^2 \le 1$). As a rule of thumb, $\eta^2 \ge 0.14$ indicates a large effect size.
>
> 2. **`continent` vs. `lifeExp`**:
>    - $\eta^2 = 0.6500$ (1952), $0.5921$ (1987), and $0.6355$ (2007).
>    - **Interpretation**: A massive **60% to 65% of all global variance in life expectancy is explained solely by which continent a country belongs to!** Geographical healthcare disparities, disease ecology, and regional stability create persistent continental clusters throughout the 55-year period.
>
> 3. **`continent` vs. `gdpPercap`**:
>    - **In 1952**: $\eta^2 = 0.0483$ (very low, <5%).
>      *Reason*: The extreme outlier Kuwait (GDP per capita \$108,382) belonged to Asia. Because Kuwait was so far above every other nation in Asia, it created gigantic **within-continent variance** in Asia ($SS_{\text{within}}$) that dwarfed between-continent variance ($SS_{\text{between}}$)!
>    - **In 1987 & 2007**: $\eta^2$ surged to **$0.4709$ and $0.4243$**.
>      *Reason*: Once outlier distortions subsided and global post-war industrialization took hold, continental grouping explained over **42% of global economic variance**, confirming profound wealth polarization between continents.
>
> -----


# **4. Time evolution**


**a.** Draw the evolution of the following correaltions from 1952 to 2007:

- `Person` and `Spearman` corerlation between life expectancy and GDP per capita
- $\eta$-squared coefficients of continent vs life expectancy, and continents vs GDP per capita.

> Hint: the following figure is expected. 

![](./corrs.png)


In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'
import plotly.express as px
import plotly.graph_objects as go

# 4.a Draw the evolution of all four correlation metrics from 1952 to 2007
time_records = []
all_years = sorted(gapminder['year'].unique())

for yr in all_years:
    sub = gapminder[gapminder['year'] == yr]
    r_pearson = sub['lifeExp'].corr(sub['gdpPercap'], method='pearson')
    r_spearman = sub['lifeExp'].corr(sub['gdpPercap'], method='spearman')
    eta_life = compute_eta_squared(sub, 'continent', 'lifeExp')
    eta_gdp = compute_eta_squared(sub, 'continent', 'gdpPercap')
    
    time_records.append({
        'year': yr,
        'Pearson (lifeExp ~ gdpPercap)': r_pearson,
        'Spearman (lifeExp ~ gdpPercap)': r_spearman,
        'eta2 (continent ~ lifeExp)': eta_life,
        'eta2 (continent ~ gdpPercap)': eta_gdp
    })

evol_df = pd.DataFrame(time_records)

# Plotting with Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=evol_df['year'], y=evol_df['Pearson (lifeExp ~ gdpPercap)'],
                         mode='lines+markers', name='Pearson (lifeExp ~ gdpPercap)',
                         line=dict(color='#1f77b4', width=2.5), marker=dict(size=8, symbol='circle')))
fig.add_trace(go.Scatter(x=evol_df['year'], y=evol_df['Spearman (lifeExp ~ gdpPercap)'],
                         mode='lines+markers', name='Spearman (lifeExp ~ gdpPercap)',
                         line=dict(color='#2ca02c', width=2.5), marker=dict(size=8, symbol='square')))
fig.add_trace(go.Scatter(x=evol_df['year'], y=evol_df['eta2 (continent ~ lifeExp)'],
                         mode='lines+markers', name='eta2 (continent ~ lifeExp)',
                         line=dict(color='#ff7f0e', width=2.5), marker=dict(size=8, symbol='triangle-up')))
fig.add_trace(go.Scatter(x=evol_df['year'], y=evol_df['eta2 (continent ~ gdpPercap)'],
                         mode='lines+markers', name='eta2 (continent ~ gdpPercap)',
                         line=dict(color='#d62728', width=2.5), marker=dict(size=8, symbol='diamond')))

fig.update_layout(
    title='Evolution of Bivariate Correlation and Association Metrics (1952–2007)',
    xaxis=dict(title='Year', tickmode='array', tickvals=all_years),
    yaxis=dict(title='Coefficient Value', range=[0, 1.0]),
    width=950,
    height=500,
    template='plotly_white',
    legend=dict(x=0.6, y=0.5, bgcolor='rgba(255,255,255,0.8)')
)
fig.show()

# Display summary table
display(evol_df.round(4))


**b.** Compare side by side the three aspects: population, life expectancy and GDP per Capita between Cambodia and Thailand from 1952 to 2007.


In [ ]:
# 4.b Side-by-side comparison: Cambodia vs. Thailand (1952 - 2007)
cam_thai = gapminder[gapminder['country'].isin(['Cambodia', 'Thailand'])].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Side-by-Side Comparison: Cambodia vs. Thailand (1952–2007)', fontsize=15, fontweight='bold')

variables = [
    ('pop', 'Population (count)', 'Population Evolution'),
    ('lifeExp', 'Life Expectancy (years)', 'Life Expectancy (Health)'),
    ('gdpPercap', 'GDP per Capita (USD)', 'GDP per Capita (Economy)')
]

colors = {'Cambodia': '#d62728', 'Thailand': '#1f77b4'}

for i, (var, ylabel, title) in enumerate(variables):
    sns.lineplot(
        data=cam_thai,
        x='year',
        y=var,
        hue='country',
        style='country',
        markers=True,
        dashes=False,
        linewidth=2.5,
        markersize=8,
        palette=colors,
        ax=axes[i]
    )
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_xlabel('Year')
    axes[i].set_ylabel(ylabel)
    axes[i].set_xticks(all_years)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


**c.** Describe what you observed: the world from 1952 to 2007.


> ### **Comprehensive Synthesis: What We Observed About the World from 1952 to 2007**
>
> 1. **Global Health Convergence (The Upward Shift of the Preston Curve)**:
>    - Global life expectancy increased dramatically across the entire world between 1952 and 2007, rising by an average of roughly 20 years.
>    - The curve relating wealth and health shifted **upward over time**: countries in 2007 achieved substantially higher life expectancy than countries with identical wealth in 1952. This was driven by the global diffusion of public health innovations such as childhood immunizations, clean drinking water systems, oral rehydration therapy, and antibiotics which extended lives even in lower-income nations.
>
> 2. **Economic Polarization Between Continents**:
>    - While health converged, **economic inequality between continents widened**.
>    - Western Europe, North America, and parts of East Asia experienced sustained industrial and technological growth, whereas Sub-Saharan Africa remained close to subsistence levels.
>    - This polarization is quantitatively verified by $\eta^2$ (continent ~ gdpPercap), which climbed from $0.048$ in 1952 to over $0.42$ in 2007, showing that continental membership became a dominant explanatory factor for national income.
>
> 3. **The Correlation Trajectories**:
>    - **Spearman rank correlation** remained very high and remarkably stable ($\rho \approx 0.77 \rightarrow 0.86$), demonstrating a robust monotonic link: higher per capita income is consistently accompanied by better health outcomes.
>    - **Pearson correlation** started artificially low in 1952 ($0.278$) due to extreme leverage outliers like Kuwait, but converged toward $0.68 - 0.75$ as global economies matured and extreme outlier leverage diminished.
>
> 4. **Tragedy and Resilience in Southeast Asia (Cambodia vs. Thailand)**:
>    - In 1952, Cambodia and Thailand shared comparable socioeconomic baselines (life expectancy around 39–50 years; GDP per capita under \$800).
>    - **Thailand** maintained peaceful institutional stability, continuous industrialization, and infrastructure expansion: life expectancy grew monotonically from $50.8$ to $70.6$ years, and GDP per capita expanded nearly tenfold to \$7,458.
>    - **Cambodia** suffered catastrophic geopolitical rupture during the 1970s Khmer Rouge regime (1975–1979). The demographic data reflects this historic tragedy with chilling clarity:
>      - Cambodia's population shrank from 7.45 million in 1972 to 6.98 million in 1977.
>      - Life expectancy collapsed to an unprecedented low of **31.2 years in 1977**.
>      - However, the post-1980 era demonstrated extraordinary national resilience: with the restoration of peace, Cambodia's life expectancy rebounded to **59.7 years**, and GDP per capita reached \$1,714 by 2007.
>
> -----
